# Experimentación PSO para CVRP (SR-1)

**Fase 2 de 3.** Corre PSO con los hiperparámetros encontrados en `1_Parametrizacion/resultados_optuna.txt` sobre las 9 instancias de evaluación, con **31 repeticiones independientes** por instancia (semillas `0..30`).

**Instancias de evaluación** (ver `chapters/4/metodo.tex`): las mismas 5 del Informe 1 (método exacto) — `E-n13-k4`, `E-n22-k4`, `E-n23-k3`, `A-n32-k5`, `A-n33-k5` — más 4 instancias más grandes fuera del alcance del exacto, para evaluar escalabilidad: `A-n60-k9`, `A-n80-k10`, `E-n76-k10`, `E-n101-k8`.

Nota: `A-n33-k5` no tiene óptimo certificado en el Informe 1 (el exacto se quedó en 667 tras 4.7h, límite de 60s/resolución); su BKS real es 661. Las demás instancias del Set A/E de este notebook usan BKS de CVRPLIB (óptimo certificado en las 4 restantes compartidas con el exacto).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import time
import numpy as np
import pandas as pd

import cvrp_common as cc

print("listo")

listo


In [2]:
# Parametros encontrados en 1_Parametrizacion (ver resultados_optuna.txt)
BEST_PARAMS = cc.PSOParams(
    n_particles=100, n_iter=500,
    w0=0.9116, wf=0.2782, c1=2.4045, c2=1.1843, v_max=0.3762,
)

EVAL_INSTANCES = [
    "E-n13-k4", "E-n22-k4", "E-n23-k3", "A-n32-k5", "A-n33-k5",   # compartidas con el metodo exacto (Informe 1)
    "A-n60-k9", "A-n80-k10", "E-n76-k10", "E-n101-k8",             # fuera del alcance del exacto (escalabilidad)
]
N_REPS = 31
SEEDS = list(range(N_REPS))

instances = {name: cc.load_instance(name) for name in EVAL_INSTANCES}
for name, inst in instances.items():
    print(f"{name}: n={inst.n_customers} m={inst.n_vehicles_ref} BKS={inst.bks} coords_exactas={inst.coords_are_exact}")

E-n13-k4: n=12 m=4 BKS=247.0 coords_exactas=False
E-n22-k4: n=21 m=4 BKS=375.0 coords_exactas=True
E-n23-k3: n=22 m=3 BKS=569.0 coords_exactas=True
A-n32-k5: n=31 m=5 BKS=784.0 coords_exactas=True
A-n33-k5: n=32 m=5 BKS=661.0 coords_exactas=True
A-n60-k9: n=59 m=9 BKS=1354.0 coords_exactas=True
A-n80-k10: n=79 m=10 BKS=1763.0 coords_exactas=True
E-n76-k10: n=75 m=10 BKS=830.0 coords_exactas=True
E-n101-k8: n=100 m=8 BKS=815.0 coords_exactas=True


In [3]:
# warmup JIT (evita que el primer run real de la tabla cargue con el costo de compilacion)
_warm = cc.load_instance("E-n13-k4")
_ = cc.run_pso(_warm, _warm.n_vehicles_ref, cc.PSOParams(n_particles=5, n_iter=2), seed=0)

rows = []
histories = {}  # (instancia, seed) -> historia de convergencia (best_cost por iteracion)

for name, inst in instances.items():
    m = inst.n_vehicles_ref
    for seed in SEEDS:
        t0 = time.perf_counter()
        res = cc.run_pso(inst, m, BEST_PARAMS, seed=seed)
        dt = time.perf_counter() - t0
        gap = cc.gap_pct(res.best_cost, inst.bks)
        rows.append({
            "instancia": name, "n": inst.n_customers, "m": m,
            "seed": seed, "costo": res.best_cost, "bks": inst.bks,
            "gap_pct": gap, "tiempo_s": dt,
        })
        histories[(name, seed)] = res.history
    print(f"{name}: listo ({N_REPS} corridas)")

df = pd.DataFrame(rows)
df.to_csv("resultados_pso.csv", index=False)
df.head()

E-n13-k4: listo (31 corridas)


E-n22-k4: listo (31 corridas)


E-n23-k3: listo (31 corridas)


A-n32-k5: listo (31 corridas)


A-n33-k5: listo (31 corridas)


A-n60-k9: listo (31 corridas)


A-n80-k10: listo (31 corridas)


E-n76-k10: listo (31 corridas)


E-n101-k8: listo (31 corridas)


,instancia,n,m,seed,costo,bks,gap_pct,tiempo_s
0,E-n13-k4,12,4,0,257.0,247.0,4.048583,0.067471
1,E-n13-k4,12,4,1,257.0,247.0,4.048583,0.065821
2,E-n13-k4,12,4,2,251.0,247.0,1.619433,0.067087
3,E-n13-k4,12,4,3,257.0,247.0,4.048583,0.065203
4,E-n13-k4,12,4,4,257.0,247.0,4.048583,0.066205


In [4]:
import pickle

with open("historiales_pso.pkl", "wb") as f:
    pickle.dump(histories, f)

summary = df.groupby("instancia").agg(
    n=("n", "first"), bks=("bks", "first"),
    costo_mejor=("costo", "min"), costo_prom=("costo", "mean"), costo_std=("costo", "std"),
    gap_mejor=("gap_pct", "min"), gap_prom=("gap_pct", "mean"), gap_std=("gap_pct", "std"),
    tiempo_prom_s=("tiempo_s", "mean"),
).reindex(EVAL_INSTANCES)
summary.to_csv("resumen_pso.csv")
summary

,n,bks,costo_mejor,costo_prom,costo_std,gap_mejor,gap_prom,gap_std,tiempo_prom_s
instancia,,,,,,,,,
E-n13-k4,12,247.0,251.0,256.612903,3.018580,1.619433,3.891864,1.222097,0.066206
E-n22-k4,21,375.0,376.0,392.483871,12.285143,0.266667,4.662366,3.276038,0.086378
E-n23-k3,22,569.0,569.0,569.193548,0.401610,0.000000,0.034016,0.070582,0.095678
A-n32-k5,31,784.0,786.0,816.709677,25.368213,0.255102,4.172153,3.235741,0.105542
A-n33-k5,32,661.0,697.0,727.580645,16.183890,5.446293,10.072715,2.448395,0.105960
A-n60-k9,59,1354.0,1486.0,1560.548387,56.342902,9.748892,15.254681,4.161219,0.172370
A-n80-k10,79,1763.0,1911.0,2045.354839,71.317388,8.394782,16.015589,4.045229,0.225964
E-n76-k10,75,830.0,924.0,1013.193548,40.275236,11.325301,22.071512,4.852438,0.211996
E-n101-k8,100,815.0,954.0,1033.709677,44.169517,17.055215,26.835543,5.419573,0.306362
